# 37 — Two-Body Measurements

**Repo:** `decoherence-free-sensing`  
**Notebook role:** measurement-protocol layer  
**Previous:** `29_quantum_fisher_specification.ipynb`  
**Next:** `43_preparation_protocols.ipynb`

Notebook 29 established the information-theoretic precision available from an admissible DFS state.

Notebook 37 asks the complementary engineering question:

**Which measurement extracts that information while preserving differential sensitivity?**

For DFS-compatible sensing, single-body coherences can vanish. A useful measurement is therefore a two-body observable such as

\[
M
=
J_x^{(A)}J_x^{(B)}
+
J_y^{(A)}J_y^{(B)}.
\]

Equivalently,

\[
M
=
\frac{1}{2}
\left(
J_A^+J_B^-
+
J_A^-J_B^+
\right).
\]

## Engineering statement

Quantum Fisher Information specifies achievable precision. Two-body measurements specify how that precision is physically realized.


## Notebook outputs

Running this notebook creates:

- `results/figures/37_two_body_measurement_specification.png`
- `results/figures/37_two_body_fringe_response.png`
- `results/figures/37_measurement_slope.png`
- `results/figures/37_error_propagation.png`
- `results/figures/37_operating_regimes.png`
- `results/two_body_measurement_summary.csv`
- `results/two_body_measurement_summary.json`
- `results/decoherence_free_sensing_two_body_measurements_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. Two-body measurement specification

Notebook 29 produced an information bound. Notebook 37 turns that bound into a measurement protocol.

The measurement pipeline is:

\[
\text{DFS state}
\rightarrow
J_z^- \text{ generator}
\rightarrow
\text{two-body observable}
\rightarrow
\text{fringe response}
\rightarrow
\text{operating point}
\rightarrow
\hat{\varphi}.
\]


In [ ]:
def draw_box(ax, xy, width, height, title, body=None, fontsize=15, facecolor="white", edgecolor="black", lw=1.8):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=lw,
        edgecolor=edgecolor,
        facecolor=facecolor
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.62 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.28,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))

fig, ax = plt.subplots(figsize=(14.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Two-Body Measurement Specification", ha="center", va="center", fontsize=23)

items = [
    (0.035, "DFS state", "$J_z^+=0$"),
    (0.225, "Generator", "$J_z^-$"),
    (0.415, "Observable", "two-body\n$M$"),
    (0.605, "Fringe", r"$\langle M\rangle(\varphi)$"),
    (0.795, "Estimate", r"operate near" + "\n" + r"$\pi/4$"),
]
w, h, y = 0.15, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body, fontsize=14)

for x0 in [0.185, 0.375, 0.565, 0.755]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "A precision resource becomes an experimental protocol only after a DFS-compatible observable is specified.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "37_two_body_measurement_specification.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 2. Two-body fringe response

A compact normalized model for the DFS-compatible two-body fringe is

\[
\langle M\rangle(\varphi)
=
-A\cos(2\varphi),
\]

where \(A\) is an amplitude set by the state and atom number.

The useful operating point occurs where the slope is largest:

\[
\varphi=\frac{\pi}{4}.
\]


In [ ]:
varphi = np.linspace(0, np.pi/2, 600)

fig, ax = plt.subplots(figsize=(8.4, 5.9))

for total_N in [40, 120, 320]:
    amplitude = (total_N**2 + 4*total_N) / 12
    response = -amplitude * np.cos(2 * varphi)
    response_normalized = response / amplitude
    ax.plot(varphi, response_normalized, linewidth=2.0, label=f"N={total_N}")

ax.axvline(np.pi/4, linestyle="--", linewidth=1.8, label=r"optimal operating point $\pi/4$")
ax.scatter([np.pi/4], [0], s=55, zorder=5)
ax.annotate(
    "maximum slope",
    xy=(np.pi/4, 0),
    xytext=(np.pi/4 + 0.10, -0.42),
    arrowprops=dict(arrowstyle="->", linewidth=1.2),
    fontsize=12
)

ax.set_title("Two-Body Fringe Response")
ax.set_xlabel(r"differential phase $\varphi$")
ax.set_ylabel(r"normalized $\langle M\rangle$")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

path = FIGURES / "37_two_body_fringe_response.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 3. Measurement slope

The local information in the fringe is governed by the slope

\[
\left|
\frac{d\langle M\rangle}{d\varphi}
\right|.
\]

For

\[
\langle M\rangle=-A\cos(2\varphi),
\]

we have

\[
\left|
\frac{d\langle M\rangle}{d\varphi}
\right|
=
2A|\sin(2\varphi)|.
\]

The slope is maximal at \(\varphi=\pi/4\).


In [ ]:
A = 1.0
response = -A * np.cos(2 * varphi)
slope = 2 * A * np.abs(np.sin(2 * varphi))

fig, ax = plt.subplots(figsize=(8.4, 5.9))
ax.plot(varphi, slope, linewidth=2.3, label=r"$|d\langle M\rangle/d\varphi|$")
ax.axvline(np.pi/4, linestyle="--", linewidth=1.8, label=r"$\pi/4$")
ax.scatter([np.pi/4], [2*A], s=55, zorder=5)

ax.set_title("Measurement Slope Peaks at the Operating Point")
ax.set_xlabel(r"differential phase $\varphi$")
ax.set_ylabel("normalized slope")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(
    0.5,
    0.32,
    "maximum phase sensitivity",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "37_measurement_slope.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. Error propagation

A measurement becomes a phase estimate through error propagation:

\[
\Delta^2\varphi
=
\frac{\Delta^2 M}
{
\left(
\frac{\partial\langle M\rangle}{\partial\varphi}
\right)^2
}.
\]

This equation explains why the operating point matters. A larger slope reduces phase variance for a fixed measurement noise.


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Error Propagation", ha="center", va="center", fontsize=23)

items = [
    (0.08, "Measurement\nvariance", r"$\Delta^2 M$"),
    (0.39, "Divide by\nslope squared", r"$|\partial_\varphi\langle M\rangle|^2$"),
    (0.70, "Phase\nuncertainty", r"$\Delta^2\varphi$"),
]
w, h, y = 0.22, 0.34, 0.40

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body, fontsize=14)

for x0 in [0.30, 0.61]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.08, y + h/2))

ax.text(
    0.5,
    0.21,
    "Maximum slope converts the two-body fringe into minimum phase uncertainty.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "37_error_propagation.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. Operating regimes

The same fringe has low-sensitivity and high-sensitivity regions.

A good protocol operates near the maximum-slope point:

\[
\varphi=\frac{\pi}{4}.
\]

Near the flat parts of the fringe, measurement outcomes change slowly with phase and provide weak local information.


In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 3.8))

phase_points = [0, np.pi/4, np.pi/2]
labels = ["low\nsensitivity", "operate here\nmaximum slope", "low\nsensitivity"]
yvals = [0, 0, 0]

ax.hlines(0, 0, np.pi/2, linewidth=2.0)
ax.scatter(phase_points, yvals, s=[70, 120, 70], zorder=5)

for p, lab in zip(phase_points, labels):
    ax.text(p, 0.14 if p != np.pi/4 else 0.22, lab, ha="center", va="bottom", fontsize=12)

ax.axvline(np.pi/4, linestyle="--", linewidth=1.5)

ax.set_xlim(-0.08, np.pi/2 + 0.08)
ax.set_ylim(-0.25, 0.55)
ax.set_yticks([])
ax.set_xticks([0, np.pi/4, np.pi/2])
ax.set_xticklabels([r"$0$", r"$\pi/4$", r"$\pi/2$"])
ax.set_xlabel(r"differential phase $\varphi$")
ax.set_title("Operating Regimes for the Two-Body Fringe")
ax.spines[["left", "right", "top"]].set_visible(False)

path = FIGURES / "37_operating_regimes.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 6. Engineering questions

| Stage | Engineering question | Notebook role |
|---|---|---|
| Observable | Which operator should be measured? | Choose a DFS-compatible two-body observable. |
| Fringe | How does expectation vary with phase? | Construct the interference response. |
| Slope | Where is information concentrated? | Identify the maximum-sensitivity operating point. |
| Error | How is phase uncertainty computed? | Apply error propagation. |
| Protocol | Where should the sensor operate? | Specify \(\varphi=\pi/4\). |


In [ ]:
summary = pd.DataFrame([
    {
        "quantity": "M = J_x^A J_x^B + J_y^A J_y^B",
        "specifies": "two-body observable",
        "engineering_meaning": "DFS-compatible measurement channel"
    },
    {
        "quantity": "<M>(varphi) = -A cos(2 varphi)",
        "specifies": "fringe response",
        "engineering_meaning": "phase-dependent observable signal"
    },
    {
        "quantity": "|d<M>/dvarphi|",
        "specifies": "local sensitivity",
        "engineering_meaning": "how quickly measurement changes with phase"
    },
    {
        "quantity": "varphi = pi/4",
        "specifies": "operating point",
        "engineering_meaning": "maximum slope and strongest local sensitivity"
    },
    {
        "quantity": "Delta^2 varphi = Delta^2 M / slope^2",
        "specifies": "error propagation",
        "engineering_meaning": "phase uncertainty from measurement noise and fringe slope"
    },
])

summary_csv = RESULTS / "two_body_measurement_summary.csv"
summary_json = RESULTS / "two_body_measurement_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 7. Key equations

Notebook 37 turns the Quantum Fisher specification into a measurement protocol.

### Two-body observable

\[
M
=
J_x^{(A)}J_x^{(B)}
+
J_y^{(A)}J_y^{(B)}.
\]

Equivalently,

\[
M
=
\frac{1}{2}
\left(
J_A^+J_B^-
+
J_A^-J_B^+
\right).
\]

### Fringe response

\[
\langle M\rangle(\varphi)
=
-A\cos(2\varphi).
\]

### Measurement slope

\[
\left|
\frac{d\langle M\rangle}
{d\varphi}
\right|
=
2A|\sin(2\varphi)|.
\]

### Error propagation

\[
\Delta^2\varphi
=
\frac{\Delta^2 M}
{
\left(
\frac{\partial\langle M\rangle}
{\partial\varphi}
\right)^2
}.
\]

### Operating point

\[
\varphi
=
\frac{\pi}{4}.
\]


## 8. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_two_body_measurements_outputs.zip"

public_figures = [
    FIGURES / "37_two_body_measurement_specification.png",
    FIGURES / "37_two_body_fringe_response.png",
    FIGURES / "37_measurement_slope.png",
    FIGURES / "37_error_propagation.png",
    FIGURES / "37_operating_regimes.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in public_figures:
        if file.exists():
            zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created: {zip_path}")

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

zip_path


## 9. Next notebook

Suggested next notebook:

`43_preparation_protocols.ipynb`

Core goal:

- compare routes for preparing useful DFS-compatible entanglement,
- connect measurement requirements to preparation requirements,
- organize unitary, dissipative, and measurement-conditioned pathways,
- close the loop from specification to implementation.
